In [ ]:
import numpy as np
import SimpleITK as sitk
from typing import Tuple
import skimage as sk
from PIL import Image
import napari
import matplotlib.pyplot as plt
import scipy as sc
from glob import glob

In [ ]:
FLIP_MATRICES = {
    "none":       [ 1,  0, 0,  1],
    "horizontal": [-1,  0, 0,  1],
    "vertical":   [1,  0, 0, -1],
    "both":       [-1,  0, 0, -1],
}
ROTATION_DICTIONARY = {
    'clockwise' : -90,
    'counter-clockwise': 90
}

In [ ]:
moving_files = sorted(glob('Masks/test_pipeline/*.png'))
fixed_files = sorted(glob('Maps/test_pipeline/*.png'))
moving_imgs = [Image.open(f).convert('L') for f in moving_files]
fixed_imgs = [Image.open(f).convert('L') for f in fixed_files]
moving_arrs = list(map(np.array,moving_imgs))
fixed_arrs = list(map(np.array,fixed_imgs))


In [ ]:
moving_rotated = sk.transform.rotate(moving_img,-90,resize=True)

In [ ]:
viewer = napari.view_image(moving_rotated)

In [ ]:
def calculate_spacing(moving,fixed,grey_value):
    #fixed = np.array(Image.open(fixed_path).convert('L'))
    #moving = np.array(Image.open(moving_path).convert('L'))
    #get largest region from tissue (excludes the cardinal points) to get bounding box of tissue mask
    moving_binary = moving > 0 
    moving_labels = sk.measure.label(moving_binary)
    moving_props = sk.measure.regionprops(moving_labels)
    largest_obj = max(moving_props, key=lambda p: p.area)
    minc,minr,maxc,maxr = largest_obj.bbox
    moving_height_px = maxr - minr
    moving_width_px  = maxc - minc
    #moving_bbox = moving_binary[minc:maxc,minr:maxr]
    #get bounding box of map region
    map_region = fixed * (fixed==grey_value)
    map_region_label = sk.measure.label(map_region)
    map_region_props = sk.measure.regionprops(map_region_label)
    minc,minr,maxc,maxr = map_region_props[0].bbox
    map_height_px = maxr - minr
    map_width_px  = maxc - minc
    # map_region_bbox = map_region[minc:maxc,minr:maxr]
    #calculate_spacing assuming pixels have 1:1 ratio
    spacing_x = map_width_px / moving_width_px
    spacing_y = map_height_px / moving_height_px
    moving_spacing = (spacing_x,spacing_y)
    map_spacing = (round((moving_width_px/map_width_px),1),round((moving_height_px/map_height_px),1))
    return map_spacing

In [ ]:
spacing = calculate_spacing(moving_rotated,fixed_img,29)
print(spacing)

In [ ]:
np.unique(mask)

In [ ]:
#moving_binary = (moving_img>0).astype(np.uint8)
labeled_mask = sk.measure.label(moving_rotated)
moving_props = sk.measure.regionprops(labeled_mask)
largest_obj = max(moving_props, key=lambda p: p.area)
label = largest_obj.label
minc,minr,maxc,maxr = largest_obj.bbox
moving_mask = ((labeled_mask==label)*moving_rotated).astype(np.float32)
moving_crop = moving_mask[minc:maxc,minr:maxr]



In [ ]:
moving_arr = sc.ndimage.binary_fill_holes(moving_crop)
moving_arr = (moving_arr).astype(np.float32)
fixed_region = fixed_img==29
fixed_binary = sc.ndimage.binary_fill_holes(fixed_region)
fixed_arr = np.array(fixed_binary).astype(np.float32)


In [ ]:
rotate = sk.transform.rotate(moving_arr,-90,resize=True)

In [ ]:
viewer = napari.view_image(rotate)

In [ ]:
moving_sitk = sitk.GetImageFromArray(moving_arr)
moving_sitk.SetSpacing((1.0,1.0))
moving_sitk.SetOrigin([0.0,0.0])

fixed_sitk = sitk.GetImageFromArray(fixed_arr)
fixed_sitk.SetSpacing(spacing)
fixed_sitk.SetOrigin([0.0,0.0])

In [ ]:
registration = sitk.ImageRegistrationMethod()
registration.SetMetricAsMeanSquares()

initial_transform = sitk.CenteredTransformInitializer(
    sitk_map, sitk_mask,
    sitk.Euler2DTransform(),
    sitk.CenteredTransformInitializerFilter.MOMENTS
)

registration.SetInitialTransform(initial_transform, inPlace=False)

registration.SetOptimizerAsLBFGS2(
solutionAccuracy=1e-5,
numberOfIterations=500,
deltaConvergenceTolerance=0.01
)

registration.SetOptimizerScalesFromPhysicalShift()
registration.SetShrinkFactorsPerLevel(shrinkFactors=[8, 4, 2, 1])
registration.SetSmoothingSigmasPerLevel(smoothingSigmas=[4, 2, 1, 0])
registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
registration.SetInterpolator(sitk.sitkLinear)

print(f"  Registering...")
final_transform = registration.Execute(sitk_map, sitk_mask)
print(f"  Metric: {registration.GetMetricValue():.6f}")

#sitk.WriteTransform(final_transform,f"test_affine_matrix_region_{label_id}.tfm")

In [ ]:
def get_cardinal_points(image_path, grey_values_dict):
    """
    Extract both cardinal points from an image.
    Returns dict with 'north' and 'east' as (x, y) arrays.
    """
    points = {}
    for direction in ["north", "east"]:
        grey = grey_values_dict[direction]
        pt = detect_cardinal_point(image_path, grey)
        if pt is None:
            raise ValueError(f"Could not find '{direction}' point in {image_path}")
        points[direction] = pt
        print(f"  {direction}: pixel ({pt[0]:.1f}, {pt[1]:.1f})")
    return points

In [ ]:
def detect_cardinal_point(image_path, grey_value):
    """
    Detect the centroid of a cardinal point marker by its grey value.
    
    Returns (x, y) in pixel coordinates, or None if not found.
    """
    arr = np.array(Image.open(image_path).convert("L"))
    
    # Create binary mask for this grey value
    mask = (arr * (arr == grey_value)).astype(int)
    
    
    if not mask.any():
        print(f"  WARNING: No pixels found for grey value {grey_value} in {image_path}")
        return None
    
    # Label connected components and get centroid
    labeled_img = sk.measure.label(mask)
    props = sk.measure.regionprops(labeled_img)
    cy,cx = props[0].centroid
    return np.array([cx, cy])  # return as (x, y)

In [ ]:
def compute_flip_transform(mask_points):
    """
    Determine and apply only the flip needed to orient the moving image
    to match the map, using the N and E cardinal point locations.
    No rotation is applied.

    Detection logic (image coordinates, y increases downward):
        North marker should be visually "above" East  → north_y < east_y
        East  marker should be visually "right" of North → east_x > north_x

    Flip cases:
        north_is_up  and east_is_right  → none
        north_is_up  and not east_is_right → horizontal  (left/right mirror)
        not north_is_up and east_is_right  → vertical    (top/bottom mirror)
        not north_is_up and not east_is_right → both

    Flip matrices are self-inverse (F @ F = I), so the forward and inverse
    transforms use the same matrix — only the translation differs.

    Parameters
    ----------
    mask_points  : dict with 'north' and 'east' as (x, y) pixel arrays — moving image

    Returns
    -------
    Flip matrix
    """
    FLIP_MATRICES = {
        "none":       [ 1,  0, 0,  1],
        "horizontal": [-1,  0, 0,  1],
        "vertical":   [1,  0, 0, -1],
        "both":       [-1,  0, 0, -1],
    }

    # --- Detect required flip from cardinal point geometry ---
    # Compare pixel coords directly for orientation — spacing does not
    # affect which direction is "up" or "right"
    north_is_up   = mask_points["north"][1] < mask_points["east"][1]  # smaller y = higher
    east_is_right = mask_points["east"][0]  > mask_points["north"][0] # larger x = righter

    if north_is_up and east_is_right:
        chosen_flip = "none"
    elif north_is_up and not east_is_right:
        chosen_flip = "horizontal"
    elif not north_is_up and east_is_right:
        chosen_flip = "vertical"
    else:
        chosen_flip = "both"

    print(f"  North is up:   {north_is_up}  "
          f"(mask north_y={mask_points['north'][1]:.1f}, east_y={mask_points['east'][1]:.1f})")
    print(f"  East is right: {east_is_right}  "
          f"(mask east_x={mask_points['east'][0]:.1f}, north_x={mask_points['north'][0]:.1f})")
    print(f"  Detected flip: '{chosen_flip}'")

    F = FLIP_MATRICES[chosen_flip]
    return F

In [ ]:
GREY_VALUES = {
    # Cardinal points
    "north": 200,   # replace with your actual grey values after
    "east":  110,   # greyscale conversion of your chosen colors
    # Region labels (map only)
    1: 29,   # e.g., blue   (0,0,255)   → L = 0.114*255 ≈ 29... 
    2: 76,  # e.g., green  (0,255,0)   → L value
    3: 150,   # e.g., red    (255,0,0)   → L value
    "exclude":  255,  # white hole
}

In [ ]:
points = get_cardinal_points(moving_path,GREY_VALUES)
flip_matrix = compute_flip_transform(points)

In [ ]:
#Registration with flip
initial_transform = sitk.CenteredTransformInitializer(fixed_arr,moving_arr,
                    sitk.AffineTransform(fixed_arr.GetDimension()),
                    sitk.CenteredTransformInitializerFilter.MOMENTS)
flip_transform = sitk.AffineTransform(fixed_arr.GetDimension())
flip_transform.SetMatrix(flip_matrix)
moving_center = moving_arr.GetOrigin() + (np.array(moving_arr.GetSpacing()) * np.array(moving_arr.GetSize()) / 2.0)
flip_transform.SetCenter(moving_center)
composite_transform = sitk.CompositeTransform(2)
composite_transform.AddTransform(initial_transform)
composite_transform.AddTransform(flip_transform)

registration_method = sitk.ImageRegistrationMethod()
registration_method.SetMetricAsMeanSquares()
registration_method.SetInitialTransform(composite_transform,inPlace=True)
registration_method.SetOptimizerAsLBFGS2(
    solutionAccuracy=1e-5,
    numberOfIterations=500,
    deltaConvergenceTolerance=0.01
    )
registration_method.SetOptimizerScalesFromPhysicalShift()
registration_method.SetShrinkFactorsPerLevel(shrinkFactors=[8, 4, 2, 1])
registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas=[4, 2, 1, 0])
registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
registration_method.SetInterpolator(sitk.sitkLinear)
#registration_method.AddCommand(sitk.sitkIterationEvent, lambda: command_iteration(registration_method))
print(f"  Registering...")
optimized_transform = registration_method.Execute(fixed_arr,moving_arr)
print(f"Metric: {registration_method.GetMetricValue():.6f}")


In [ ]:
#Registration with rotation
angle = (np.deg2rad(-90),)
initial_transform = sitk.CenteredTransformInitializer(fixed_sitk,moving_sitk,
                    sitk.AffineTransform(2),
                    sitk.CenteredTransformInitializerFilter.MOMENTS)
rotation_transform = sitk.Euler2DTransform()
moving_center = moving_sitk.GetOrigin() + (np.array(moving_sitk.GetSpacing()) * np.array(moving_sitk.GetSize()) / 2.0)
rotation_transform.SetCenter(moving_center)
rotation_transform.SetAngle(angle[0])
composite_transform = sitk.CompositeTransform(2)
composite_transform.AddTransform(initial_transform)
#composite_transform.AddTransform(rotation_transform)

registration_method = sitk.ImageRegistrationMethod()
registration_method.SetMetricAsCorrelation()
registration_method.SetInitialTransform(composite_transform,inPlace=True)
registration_method.SetOptimizerAsLBFGS2(
    solutionAccuracy=1e-5,
    numberOfIterations=500,
    deltaConvergenceTolerance=0.01
    )
registration_method.SetOptimizerScalesFromPhysicalShift()
registration_method.SetShrinkFactorsPerLevel(shrinkFactors=[8, 4, 2, 1])
registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas=[4, 2, 1, 0])
registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
registration_method.SetInterpolator(sitk.sitkNearestNeighbor)
#registration_method.AddCommand(sitk.sitkIterationEvent, lambda: command_iteration(registration_method))
print(f"  Registering...")
optimized_transform = registration_method.Execute(fixed_sitk,moving_sitk)
print(f"Metric: {registration_method.GetMetricValue():.6f}")


Try adding in a resampling step after the euler transform to create a new image to prevent the mask from getting cut off (use the size of the map?)
Or apply rotation through affine?
last idea - add padding based on longest axis

In [ ]:
#Registration with rotation
angle = (np.deg2rad(-90),)
initial_transform = sitk.CenteredTransformInitializer(fixed_sitk,moving_sitk,
                    sitk.Euler2DTransform(),
                    sitk.CenteredTransformInitializerFilter.MOMENTS)
rotation_transform = sitk.Euler2DTransform()
moving_center = moving_sitk.GetOrigin() + (np.array(moving_sitk.GetSpacing()) * np.array(moving_sitk.GetSize()) / 2.0)
rotation_transform.SetCenter(moving_center)
rotation_transform.SetAngle(angle[0])
composite_transform = sitk.CompositeTransform(2)
composite_transform.AddTransform(initial_transform)
#composite_transform.AddTransform(rotation_transform)

registration_method = sitk.ImageRegistrationMethod()
registration_method.SetMetricAsMeanSquares()
registration_method.SetInitialTransform(composite_transform,inPlace=True)
registration_method.SetOptimizerAsExhaustive([6//2,0,0])
registration_method.SetOptimizerScales([2.0*np.pi/6,1.0,1.0])
registration_method.SetInterpolator(sitk.sitkNearestNeighbor)
#registration_method.AddCommand(sitk.sitkIterationEvent, lambda: command_iteration(registration_method))
print(f"  Registering...")
optimized_transform = registration_method.Execute(fixed_sitk,moving_sitk)
print(f"Metric: {registration_method.GetMetricValue():.6f}")


In [ ]:
resampled = sitk.Resample(
        moving_sitk, fixed_sitk, optimized_transform,
        sitk.sitkNearestNeighbor, 0.0, moving_sitk.GetPixelID()
    )
fig, axes = plt.subplots()
fixed   = sitk.GetArrayFromImage(fixed_sitk)
moved   = sitk.GetArrayFromImage(resampled)
axes.imshow(fixed, cmap="gray")
axes.imshow(moved,cmap="Reds",alpha=0.5)

plt.tight_layout()
#plt.savefig("registration_verification_fullmap.png", dpi=150)
plt.show()

In [ ]:
flip_transform = sitk.AffineTransform(fixed_arr.GetDimension())
flip_transform.SetMatrix(flip_matrix)

center_transform = sitk.CenteredTransformInitializer(fixed_arr,moving_arr,sitk.AffineTransform(fixed_arr.GetDimension()),sitk.CenteredTransformInitializerFilter.MOMENTS)
registration_method = sitk.ImageRegistrationMethod()
registration_method.SetMetricAsMeanSquares()
registration_method.SetInitialTransform(center_transform)
registration_method.SetOptimizerAsLBFGS2(
    solutionAccuracy=1e-5,
    numberOfIterations=500,
    deltaConvergenceTolerance=0.01
    )
registration_method.SetOptimizerScalesFromPhysicalShift()
registration_method.SetShrinkFactorsPerLevel(shrinkFactors=[8, 4, 2, 1])
registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas=[4, 2, 1, 0])
registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
registration_method.SetInterpolator(sitk.sitkLinear)
#registration_method.AddCommand(sitk.sitkIterationEvent, lambda: command_iteration(registration_method))
optimized_transform = registration_method.Execute(fixed_arr,moving_arr)
composite_transform = sitk.CompositeTransform(optimized_transform)
composite_transform.AddTransform(flip_transform)
composite_transform.WriteTransform('test_composite.tfm')
composite_transform.GetMe
print(f"Metric: {registration_method.GetMetricValue():.6f}")


In [ ]:
def multi_start_registration(fixed, moving, flip_matrix, n_rotations=4):
    """
    Try N evenly-spaced initial rotations, run full registration from each,
    return the transform with the best final metric.
    """
    best_metric    = np.inf
    best_transform = None
    
    angles = np.linspace(0, 2 * np.pi, n_rotations, endpoint=False)
    
    for i, angle in enumerate(angles):
        flip = flip_matrix
        affine = sitk.AffineTransform(2)
        affine.SetMatrix(flip.flatten().tolist())
        # Build rotated initial transform
        init = sitk.CenteredTransformInitializer(
            fixed, moving, sitk.AffineTransform(2),
            sitk.CenteredTransformInitializerFilter.MOMENTS
        )
        affine.SetCenter(init.GetFixedParameters()[:2])
        #apply flip

        # Apply rotation offset
        rot = np.array([[np.cos(angle), -np.sin(angle)],
                        [np.sin(angle),  np.cos(angle)]])
        affine.SetMatrix(rot.flatten().tolist())
        affine.SetTranslation(init.GetTranslation())
        
        reg = sitk.ImageRegistrationMethod()
        reg.SetMetricAsMeanSquares()
        reg.SetInitialTransform(affine, inPlace=False)
        reg.SetOptimizerAsGradientDescent(
            learningRate=1.0, numberOfIterations=300,
            convergenceMinimumValue=1e-6, convergenceWindowSize=10
        )
        reg.SetOptimizerScalesFromPhysicalShift()
        reg.SetShrinkFactorsPerLevel(shrinkFactors=[4, 2, 1])
        reg.SetSmoothingSigmasPerLevel(smoothingSigmas=[2, 1, 0])
        reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
        reg.SetInterpolator(sitk.sitkLinear)
        
        try:
            result = reg.Execute(fixed, moving)
            metric = reg.GetMetricValue()
            print(f"  Start {i+1}/{n_rotations} (angle={np.degrees(angle):.0f}°) metric: {metric:.6f}")
            
            if metric < best_metric:
                best_metric    = metric
                best_transform = result
        except Exception as e:
            print(f"  Start {i+1} failed: {e}")
    
    print(f"  Best metric: {best_metric:.6f}")
    return best_transform

In [ ]:
GREY_VALUES = {
    # Cardinal points
    "north": 200,   # replace with your actual grey values after
    "east":  110,   # greyscale conversion of your chosen colors
    # Region labels (map only)
    1: 29,   # e.g., blue   (0,0,255)   → L = 0.114*255 ≈ 29... 
    2: 76,  # e.g., green  (0,255,0)   → L value
    3: 150,   # e.g., red    (255,0,0)   → L value
    "exclude":  255,  # white hole
}

In [ ]:
points = get_cardinal_points('Masks/Tissue1/example_mask.png',GREY_VALUES)

In [ ]:
F = compute_flip_transform(points)

In [ ]:
best_transform = multi_start_registration(sitk_map,sitk_mask,flip_matrix=F,n_rotations=4)

In [ ]:
transform = sitk.ReadTransform()